<a href="https://colab.research.google.com/github/akinns247/Starter_Notebook247/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akinns247/Starter_Notebook247/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
Method: Random Forest Classifier

I chose a Random Forest because my lane is content refresh prioritization and I need to identify which pages are more likely to belong to the refresh-priority group. Random Forest can capture nonlinear relationships between historical SEO and engagement signals and can produce prediction probabilities that can be used to rank pages. It is also a reasonable model for comparison against my simple Week-4 baseline because it adds learned relationships without relying only on one hand-built rule.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
I use the March 2026 slice because my data contract selected it as a mid-panel decision month rather than the sealed final month.

Because each client can have multiple pages, I use a grouped train/test split by `client_id`. This keeps pages from the same client from appearing in both training and test data. That gives a more honest estimate of how the model may generalize to unseen clients.

The baseline and the Random Forest are evaluated on the exact same held-out test rows and use the same ranking metric, Precision@20. Thresholds used by the baseline are calculated from the training portion only so the test set is not used to tune the rule.

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit

# ---------------------------------------------------------
# 1. Load the real dataset
# ---------------------------------------------------------

possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("/content/Starter_Notebook247/data/raw/content_refresh_anonymized.csv"),
    Path("/content/data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in possible_paths if p.exists()), None)

if data_path is not None:
    df = pd.read_csv(data_path)
    print("Loaded dataset from:", data_path)
else:
    data_url = (
        "https://raw.githubusercontent.com/akinns247/"
        "Starter_Notebook247/main/data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(data_url)
    print("Loaded dataset from GitHub.")

print("Full dataset shape:", df.shape)

# ---------------------------------------------------------
# 2. Check required columns
# ---------------------------------------------------------

required_columns = [
    "content_id",
    "client_id",
    "ctr",
    "avg_position",
    "trend_direction",
    "search_volume",
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise KeyError(f"Missing required columns: {missing}")

print("\nRequired columns found successfully.")

# ---------------------------------------------------------
# 3. Clean the data
# ---------------------------------------------------------

df["trend_direction"] = (
    df["trend_direction"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df = df.dropna(
    subset=[
        "client_id",
        "ctr",
        "avg_position",
        "trend_direction",
        "search_volume",
    ]
).copy()

print("Rows after cleaning:", len(df))

# ---------------------------------------------------------
# 4. Select model features
# ---------------------------------------------------------

numeric_candidates = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_candidates = [
    "content_type",
    "main_intent",
    "competition_level",
]

numeric_features = [
    c for c in numeric_candidates
    if c in df.columns
]

categorical_features = [
    c for c in categorical_candidates
    if c in df.columns
]

model_features = numeric_features + categorical_features

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTotal model features:", len(model_features))

# ---------------------------------------------------------
# 5. Convert numeric features
# ---------------------------------------------------------

for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ---------------------------------------------------------
# 6. Grouped train/test split by client
# ---------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("\nTraining rows:", len(train))
print("Test rows:", len(test))

print("\nTraining clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

# ---------------------------------------------------------
# 7. Create proxy target using TRAIN thresholds only
# ---------------------------------------------------------

ctr_cutoff = train["ctr"].quantile(0.25)
position_cutoff = train["avg_position"].quantile(0.75)

train["needs_refresh"] = (
    (train["ctr"] <= ctr_cutoff)
    & (train["trend_direction"] == "down")
    & (train["avg_position"] >= position_cutoff)
).astype(int)

test["needs_refresh"] = (
    (test["ctr"] <= ctr_cutoff)
    & (test["trend_direction"] == "down")
    & (test["avg_position"] >= position_cutoff)
).astype(int)

print("\nProxy target counts — training:")
print(train["needs_refresh"].value_counts())

print("\nProxy target counts — test:")
print(test["needs_refresh"].value_counts())

print("\nTraining thresholds:")
print("CTR cutoff:", round(ctr_cutoff, 6))
print("Position cutoff:", round(position_cutoff, 4))

# ---------------------------------------------------------
# 8. Check that clients do not overlap
# ---------------------------------------------------------

overlap = set(train["client_id"]).intersection(
    set(test["client_id"])
)

print("\nClient overlap:", len(overlap))

assert len(overlap) == 0, (
    "ERROR: Some clients appear in both train and test."
)

print("Grouped split check: PASSED")

Loaded dataset from GitHub.
Full dataset shape: (30000, 44)

Required columns found successfully.
Rows after cleaning: 27532

Numeric features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Categorical features:
['content_type', 'main_intent', 'competition_level']

Total model features: 11

Training rows: 21884
Test rows: 5648

Training clients: 24
Test clients: 7

Proxy target counts — training:
needs_refresh
0    20455
1     1429
Name: count, dtype: int64

Proxy target counts — test:
needs_refresh
0    5478
1     170
Name: count, dtype: int64

Training thresholds:
CTR cutoff: 0.0
Position cutoff: 24.8

Client overlap: 0
Grouped split check: PASSED


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*
## 3. Train + compare vs my baseline

I train a Random Forest classifier using the training clients only. The model probability for `needs_refresh` is used as the ranking score.

I recreate my Week-4 baseline on the same held-out test rows. The baseline uses the CTR opportunity signal and search-demand signal from my earlier rule.

Both methods are evaluated on the same test data using Precision@20 as the main metric, because the practical goal is to identify a small set of pages for review first.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import os

# ---------------------------------------------------------
# 1. Training and test data
# ---------------------------------------------------------

X_train = train[model_features].copy()
X_test = test[model_features].copy()

y_train = train["needs_refresh"].copy()
y_test = test["needs_refresh"].copy()

# ---------------------------------------------------------
# 2. Preprocessing
# ---------------------------------------------------------

transformers = []

if numeric_features:
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]
    )

    transformers.append(
        ("num", numeric_transformer, numeric_features)
    )

if categorical_features:
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )

    transformers.append(
        ("cat", categorical_transformer, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop"
)

# ---------------------------------------------------------
# 3. Random Forest model
# ---------------------------------------------------------

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf)
    ]
)

# Train the model
model.fit(X_train, y_train)

# Probability becomes the ranking score
model_scores = model.predict_proba(X_test)[:, 1]

print("Random Forest training completed.")

# ---------------------------------------------------------
# 4. Recreate the Week-4 baseline
# ---------------------------------------------------------

train_ctr_median = train["ctr"].median()
train_search_median = train["search_volume"].median()

baseline_score = np.zeros(len(test))

ctr_opportunity = (
    (test["ctr"] < train_ctr_median)
    & (test["avg_position"] <= 10)
)

high_search_demand = (
    test["search_volume"] > train_search_median
)

baseline_score += ctr_opportunity.astype(int) * 50
baseline_score += high_search_demand.astype(int) * 30

# ---------------------------------------------------------
# 5. Precision@20 helper
# ---------------------------------------------------------

def precision_at_k(y_true, scores, k=20):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_indices].sum() / k


def ranking_metrics(y_true, scores, k=20):
    precision_k = precision_at_k(y_true, scores, k)

    predictions = (
        np.asarray(scores) >= 0.50
    ).astype(int)

    precision = precision_score(
        y_true,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        predictions,
        zero_division=0
    )

    auc = roc_auc_score(
        y_true,
        scores
    )

    return {
        "Precision@20": precision_k,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": auc
    }


# ---------------------------------------------------------
# 6. Compare model and baseline
# ---------------------------------------------------------

model_metrics = ranking_metrics(
    y_test,
    model_scores,
    k=20
)

baseline_metrics = ranking_metrics(
    y_test,
    baseline_score,
    k=20
)

comparison = pd.DataFrame([
    {
        "Approach": "Week-4 baseline",
        **baseline_metrics
    },
    {
        "Approach": "Random Forest",
        **model_metrics
    }
])

comparison[
    [
        "Precision@20",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ]
] = comparison[
    [
        "Precision@20",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ]
].round(4)

print("\nMODEL VS BASELINE")
display(comparison)

# ---------------------------------------------------------
# 7. Show top-ranked pages
# ---------------------------------------------------------

results = test[
    [
        "content_id",
        "search_volume",
        "ctr",
        "avg_position",
        "trend_direction",
        "needs_refresh"
    ]
].copy()

results["model_score"] = model_scores
results["baseline_score"] = baseline_score

print("\nTop 20 pages ranked by Random Forest:")
display(
    results.sort_values(
        "model_score",
        ascending=False
    ).head(20)
)

print("\nTop 20 pages ranked by Week-4 baseline:")
display(
    results.sort_values(
        ["baseline_score", "search_volume"],
        ascending=[False, False]
    ).head(20)
)

# ---------------------------------------------------------
# 8. Save outputs
# ---------------------------------------------------------

os.makedirs("work/outputs", exist_ok=True)

comparison.to_csv(
    "work/outputs/ml08_model_vs_baseline.csv",
    index=False
)

results.sort_values(
    "model_score",
    ascending=False
).to_csv(
    "work/outputs/ml08_test_rankings.csv",
    index=False
)

print("\nOutput files saved.")

Random Forest training completed.

MODEL VS BASELINE


,Approach,Precision@20,Precision,Recall,F1,ROC-AUC
0,Week-4 baseline,0.00,0.0237,0.4882,0.0453,0.3439
1,Random Forest,0.05,0.0358,0.2647,0.0630,0.5682



Top 20 pages ranked by Random Forest:


,content_id,search_volume,ctr,avg_position,trend_direction,needs_refresh,model_score,baseline_score
883,content_ee66da1b607c,10.0,0.00,46.5,stable,0,0.814156,0.0
11445,content_1a7560472af6,170.0,0.00,0.0,new,0,0.814108,80.0
20450,content_1fc1c9d18ad6,10.0,0.00,4.0,flat,0,0.741065,50.0
10185,content_b12742b07c1d,10.0,0.00,2.0,flat,0,0.739356,50.0
551,content_83485ea6300e,10.0,0.00,71.0,flat,0,0.738126,0.0
16851,content_141704f4d910,10.0,0.00,1.0,new,0,0.736668,50.0
19244,content_e8f44fac6055,10.0,0.00,0.0,new,0,0.732289,50.0
23325,content_1a6fbd3bbcd5,30.0,0.00,0.7,flat,0,0.731308,80.0
25560,content_1d2233dc3323,20.0,0.00,1.5,up,0,0.726752,80.0
15639,content_cb6c7d58c0bc,10.0,0.00,144.5,new,0,0.726737,0.0



Top 20 pages ranked by Week-4 baseline:


,content_id,search_volume,ctr,avg_position,trend_direction,needs_refresh,model_score,baseline_score
2074,content_6ef3dcb7be11,27100.0,0.07,6.0,down,0,0.441062,80.0
17859,content_5c5fab9d41e7,22200.0,0.06,5.4,up,0,0.469342,80.0
2754,content_b925c292d21b,14800.0,0.00,7.1,down,0,0.436152,80.0
16736,content_e12868d1f396,12100.0,0.07,2.9,stable,0,0.245075,80.0
25772,content_2e0b3dc70916,9900.0,0.04,6.9,stable,0,0.290847,80.0
21867,content_608540486d95,8100.0,0.00,7.0,down,0,0.420822,80.0
3484,content_5dcb74bcde1f,6600.0,0.00,6.5,flat,0,0.277507,80.0
9851,content_2725d2bcfac1,6600.0,0.02,9.1,down,0,0.360372,80.0
29161,content_33ddda1a8c7a,6600.0,0.00,6.9,down,0,0.384924,80.0
11883,content_137b16f29ca9,5400.0,0.00,0.0,new,0,0.583768,80.0



Output files saved.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
## 4. Errors and interpretation

I inspect false positives and false negatives to understand where the model disagrees with the proxy target.

A false positive is a page the model predicts as refresh-priority even though the proxy label is 0. A false negative is a page with proxy label 1 that the model gives a lower prediction.

I also use permutation importance to see which available features most affect the model's performance. These results are directional and describe model behavior; they do not prove that a feature causes a page to need refreshing.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

# ---------------------------------------------------------
# 1. Model predictions
# ---------------------------------------------------------

model_predictions = (
    model_scores >= 0.50
).astype(int)

error_table = test[
    [
        "content_id",
        "search_volume",
        "ctr",
        "avg_position",
        "trend_direction",
        "needs_refresh"
    ]
].copy()

error_table["model_score"] = model_scores
error_table["model_prediction"] = model_predictions

# ---------------------------------------------------------
# 2. False positives
# ---------------------------------------------------------

false_positives = error_table[
    (error_table["needs_refresh"] == 0) &
    (error_table["model_prediction"] == 1)
].sort_values(
    "model_score",
    ascending=False
)

# ---------------------------------------------------------
# 3. False negatives
# ---------------------------------------------------------

false_negatives = error_table[
    (error_table["needs_refresh"] == 1) &
    (error_table["model_prediction"] == 0)
].sort_values(
    "model_score",
    ascending=False
)

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nSample false positives:")
display(false_positives.head(10))

print("\nSample false negatives:")
display(false_negatives.head(10))

# ---------------------------------------------------------
# 4. Permutation importance
# ---------------------------------------------------------

permutation = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=3,
    random_state=42,
    scoring="roc_auc",
    n_jobs=-1
)

importance = pd.Series(
    permutation.importances_mean,
    index=X_test.columns
).sort_values(
    ascending=False
)

print("\nTop permutation-importance features:")
display(
    importance.head(10).to_frame(
        "mean_importance"
    ).round(5)
)

# ---------------------------------------------------------
# 5. Leakage check
# ---------------------------------------------------------

direct_target_inputs = {
    "ctr",
    "avg_position",
    "trend_direction",
    "clicks",
    "impressions"
}

leaked_features = direct_target_inputs.intersection(
    set(model_features)
)

print("\nDirect target-related features used by model:")
print(leaked_features)

assert len(leaked_features) == 0, (
    "Leakage check failed. A direct target ingredient "
    "is being used as a model feature."
)

# IDs and administrative fields should not be model features.
assert "client_id" not in model_features
assert "content_id" not in model_features

print("\nLeakage check: PASSED")
print("Client/content identifiers used as features: NO")

False positives: 1213
False negatives: 125

Sample false positives:


,content_id,search_volume,ctr,avg_position,trend_direction,needs_refresh,model_score,model_prediction
883,content_ee66da1b607c,10.0,0.0,46.5,stable,0,0.814156,1
11445,content_1a7560472af6,170.0,0.0,0.0,new,0,0.814108,1
20450,content_1fc1c9d18ad6,10.0,0.0,4.0,flat,0,0.741065,1
10185,content_b12742b07c1d,10.0,0.0,2.0,flat,0,0.739356,1
551,content_83485ea6300e,10.0,0.0,71.0,flat,0,0.738126,1
16851,content_141704f4d910,10.0,0.0,1.0,new,0,0.736668,1
19244,content_e8f44fac6055,10.0,0.0,0.0,new,0,0.732289,1
23325,content_1a6fbd3bbcd5,30.0,0.0,0.7,flat,0,0.731308,1
25560,content_1d2233dc3323,20.0,0.0,1.5,up,0,0.726752,1
15639,content_cb6c7d58c0bc,10.0,0.0,144.5,new,0,0.726737,1



Sample false negatives:


,content_id,search_volume,ctr,avg_position,trend_direction,needs_refresh,model_score,model_prediction
12243,content_ab58ad627e6b,390.0,0.0,29.7,down,1,0.495987,0
13064,content_d81093249cd6,0.0,0.0,26.9,down,1,0.487129,0
25576,content_d4d880bb41f3,40.0,0.0,25.5,down,1,0.486070,0
1563,content_8ac5e7d55b90,10.0,0.0,52.2,down,1,0.484904,0
2208,content_827b209fa167,10.0,0.0,45.0,down,1,0.481212,0
7637,content_a080bd4f0f51,10.0,0.0,32.4,down,1,0.481212,0
21909,content_bc805497f866,20.0,0.0,47.0,down,1,0.479346,0
28482,content_5292478e83f6,20.0,0.0,47.1,down,1,0.479346,0
808,content_1b259c0f9659,0.0,0.0,33.5,down,1,0.475793,0
1318,content_576d85a2a756,10.0,0.0,46.2,down,1,0.474874,0



Top permutation-importance features:


,mean_importance
engagement_rate,0.03635
scroll_rate,0.02940
char_count,0.01942
competition,0.00918
word_count,0.00549
competition_level,0.00547
content_type,0.00000
ai_traffic_pct,-0.00034
main_intent,-0.00221
search_volume,-0.00655



Direct target-related features used by model:
set()

Leakage check: PASSED
Client/content identifiers used as features: NO


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.